# 🧠 NLP Basics: SMS Spam Detection

Natural Language Processing (NLP) is the branch of AI that deals with giving computers the ability to understand text and spoken words.

This notebook walks you through the core concepts of NLP, explaining exactly how we take raw human text messages (like *"Win a free iPhone!"*) and convert them into math so a Machine Learning model can detect spam.

## 1. The Core Problem

Machine Learning models (like Random Forests, Neural Networks, or Naive Bayes) are mathematical algorithms. They only understand numbers. If you try to feed the string `"Hello"` into a model, it will crash.

**NLP** is the bridge. It is the process of extracting features from text and turning them into a numerical matrix.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Load the dataset
# The dataset is a TSV (Tab-Separated Values) file without headers.
df = pd.read_csv("dataset/SMSSpamCollection", sep="\t", names=["label", "message"])
print(f"Total messages: {len(df)}")
df.head()

## 2. Text Preprocessing & TF-IDF (The Math)

Before we do math, we have to clean the text using Tokenization, Lowercasing, and Stop Word Removal (removing words like "the", "is").

Then we use **TF-IDF (Term Frequency-Inverse Document Frequency)** to score the words. If a word like "FREE" appears rarely overall, but heavily in one message, it gets a massive mathematical score!

In [ ]:
# Convert labels to math (0 or 1)
df["label"] = df["label"].map({"ham": 0, "spam": 1})

# Train/Test Split
X = df["message"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# The TF-IDF Vectorizer automatically handles Tokenization, Lowercasing, and Stop Words!
vectorizer = TfidfVectorizer(stop_words="english")
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())} unique words")

## 3. The Algorithm: Multinomial Naive Bayes

For NLP classification, **Multinomial Naive Bayes** is the industry standard baseline. It calculates probabilities based on Bayes Theorem: *"What is the probability this message is SPAM, given that it contains the word prize?"*

In [ ]:
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

predictions = model.predict(X_test_tfidf)
acc = accuracy_score(y_test, predictions)
print(f"Overall Accuracy: {acc * 100:.2f}%")
print(classification_report(y_test, predictions, target_names=["Ham", "Spam"]))

## 4. Test It Yourself!
Try adding your own text messages below.

In [ ]:
custom_messages = [
    "Hey man, are we still going to the game tonight?",
    "URGENT! You have won a 1 week FREE membership in our £100,000 Prize Jackpot! Txt the word: CLAIM to No: 81010"
]

# Must transform using the EXACT SAME vectorizer we trained earlier
custom_tfidf = vectorizer.transform(custom_messages)
custom_preds = model.predict(custom_tfidf)

for msg, pred in zip(custom_messages, custom_preds):
    label = "🚨 SPAM" if pred == 1 else "✅ HAM"
    print(f"[{label}] -> {msg}")